# OCR Pipeline untuk Aksara Jawa dan Sunda

Pipeline ini menggunakan pendekatan transfer learning dengan EfficientNetB0 untuk klasifikasi aksara.

## Struktur Pipeline:
1. **Data Loading & Preprocessing**: Resize, normalisasi, augmentasi
2. **Model Architecture**: Transfer learning dengan EfficientNetB0
3. **Training**: Terpisah untuk aksara Jawa dan Sunda
4. **Evaluation & Prediction**: Dengan confidence thresholding

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import cv2

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration & Hyperparameters

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

JAWA_TRAIN_DIR = 'dataset-aksara-jawa/train'
JAWA_VAL_DIR = 'dataset-aksara-jawa/val'
SUNDA_DIR = 'dataset-aksara-sunda'

MODEL_JAWA_PATH = 'model_aksara_jawa.h5'
MODEL_SUNDA_PATH = 'model_aksara_sunda.h5'

## 3. Preprocessing Functions

In [ ]:
def preprocess_image(img_path, target_size=(224, 224)):
    """
    Preprocessing image: grayscale conversion, resize, normalization
    """
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, target_size)
    
    img = cv2.GaussianBlur(img, (3, 3), 0)
    
    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    img = np.stack([img] * 3, axis=-1)
    img = img / 255.0
    
    return img

def load_dataset(data_dir, target_size=(224, 224)):
    """
    Load dataset dari direktori dengan struktur folder per kelas
    """
    images = []
    labels = []
    class_names = sorted(os.listdir(data_dir))
    class_names = [c for c in class_names if os.path.isdir(os.path.join(data_dir, c))]
    
    for class_idx, class_name in enumerate(class_names):
        class_path = os.path.join(data_dir, class_name)
        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(class_path, img_name)
                try:
                    img = preprocess_image(img_path, target_size)
                    images.append(img)
                    labels.append(class_idx)
                except Exception as e:
                    continue
    
    return np.array(images), np.array(labels), class_names

## 4. Model Architecture dengan Transfer Learning

In [ ]:
def build_model(num_classes, input_shape=(224, 224, 3)):
    """
    Build model menggunakan EfficientNetB0 dengan transfer learning
    """
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    
    base_model.trainable = False
    
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model, base_model

## 5. Training Pipeline untuk Aksara Jawa

In [ ]:
print("Loading Aksara Jawa dataset...")
X_train_jawa, y_train_jawa, class_names_jawa = load_dataset(JAWA_TRAIN_DIR, (IMG_SIZE, IMG_SIZE))
X_val_jawa, y_val_jawa, _ = load_dataset(JAWA_VAL_DIR, (IMG_SIZE, IMG_SIZE))

print(f"Jawa Train: {X_train_jawa.shape}, Val: {X_val_jawa.shape}")
print(f"Jumlah kelas: {len(class_names_jawa)}")
print(f"Kelas: {class_names_jawa}")

In [ ]:
model_jawa, base_model_jawa = build_model(len(class_names_jawa))
print(f"Model Aksara Jawa created with {len(class_names_jawa)} classes")

callbacks_jawa = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint(MODEL_JAWA_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
]

In [ ]:
print("Training Aksara Jawa model (Phase 1: Frozen base)...")
history_jawa_1 = model_jawa.fit(
    X_train_jawa, y_train_jawa,
    validation_data=(X_val_jawa, y_val_jawa),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_jawa,
    verbose=1
)

In [ ]:
print("Fine-tuning Aksara Jawa model (Phase 2: Unfrozen layers)...")
base_model_jawa.trainable = True

for layer in base_model_jawa.layers[:-20]:
    layer.trainable = False

model_jawa.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_jawa_2 = model_jawa.fit(
    X_train_jawa, y_train_jawa,
    validation_data=(X_val_jawa, y_val_jawa),
    epochs=30,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_jawa,
    verbose=1
)

## 6. Training Pipeline untuk Aksara Sunda

In [ ]:
print("Loading Aksara Sunda dataset...")
X_sunda, y_sunda, class_names_sunda = load_dataset(SUNDA_DIR, (IMG_SIZE, IMG_SIZE))

X_train_sunda, X_val_sunda, y_train_sunda, y_val_sunda = train_test_split(
    X_sunda, y_sunda, test_size=0.2, random_state=42, stratify=y_sunda
)

print(f"Sunda Train: {X_train_sunda.shape}, Val: {X_val_sunda.shape}")
print(f"Jumlah kelas: {len(class_names_sunda)}")
print(f"Kelas: {class_names_sunda}")

In [ ]:
model_sunda, base_model_sunda = build_model(len(class_names_sunda))
print(f"Model Aksara Sunda created with {len(class_names_sunda)} classes")

callbacks_sunda = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint(MODEL_SUNDA_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
]

In [ ]:
print("Training Aksara Sunda model (Phase 1: Frozen base)...")
history_sunda_1 = model_sunda.fit(
    X_train_sunda, y_train_sunda,
    validation_data=(X_val_sunda, y_val_sunda),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_sunda,
    verbose=1
)

In [ ]:
print("Fine-tuning Aksara Sunda model (Phase 2: Unfrozen layers)...")
base_model_sunda.trainable = True

for layer in base_model_sunda.layers[:-20]:
    layer.trainable = False

model_sunda.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_sunda_2 = model_sunda.fit(
    X_train_sunda, y_train_sunda,
    validation_data=(X_val_sunda, y_val_sunda),
    epochs=30,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_sunda,
    verbose=1
)

## 7. Evaluation & Visualization

In [ ]:
def plot_training_history(history1, history2, title):
    """
    Plot training history untuk kedua fase training
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs1 = range(1, len(history1.history['accuracy']) + 1)
    epochs2 = range(len(epochs1) + 1, len(epochs1) + len(history2.history['accuracy']) + 1)
    
    axes[0].plot(epochs1, history1.history['accuracy'], 'b-', label='Train Phase 1')
    axes[0].plot(epochs1, history1.history['val_accuracy'], 'b--', label='Val Phase 1')
    axes[0].plot(epochs2, history2.history['accuracy'], 'r-', label='Train Phase 2')
    axes[0].plot(epochs2, history2.history['val_accuracy'], 'r--', label='Val Phase 2')
    axes[0].set_title(f'{title} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(epochs1, history1.history['loss'], 'b-', label='Train Phase 1')
    axes[1].plot(epochs1, history1.history['val_loss'], 'b--', label='Val Phase 1')
    axes[1].plot(epochs2, history2.history['loss'], 'r-', label='Train Phase 2')
    axes[1].plot(epochs2, history2.history['val_loss'], 'r--', label='Val Phase 2')
    axes[1].set_title(f'{title} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history_jawa_1, history_jawa_2, 'Aksara Jawa')
plot_training_history(history_sunda_1, history_sunda_2, 'Aksara Sunda')

In [ ]:
def evaluate_model(model, X_val, y_val, class_names, title):
    """
    Evaluate model dan tampilkan metrics
    """
    y_pred_proba = model.predict(X_val)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    print(f"\n{'='*60}")
    print(f"Evaluation Results - {title}")
    print(f"{'='*60}")
    print(classification_report(y_val, y_pred, target_names=class_names))
    
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {title}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    return y_pred, y_pred_proba

y_pred_jawa, y_pred_proba_jawa = evaluate_model(
    model_jawa, X_val_jawa, y_val_jawa, class_names_jawa, 'Aksara Jawa'
)

y_pred_sunda, y_pred_proba_sunda = evaluate_model(
    model_sunda, X_val_sunda, y_val_sunda, class_names_sunda, 'Aksara Sunda'
)

## 8. Prediction & Post-processing

In [ ]:
def predict_single_image(img_path, model, class_names, confidence_threshold=0.5):
    """
    Prediksi single image dengan confidence thresholding
    """
    img = preprocess_image(img_path, (IMG_SIZE, IMG_SIZE))
    img_batch = np.expand_dims(img, axis=0)
    
    predictions = model.predict(img_batch, verbose=0)
    confidence = np.max(predictions)
    predicted_class_idx = np.argmax(predictions)
    predicted_class = class_names[predicted_class_idx]
    
    if confidence < confidence_threshold:
        return None, confidence, "Low confidence"
    
    return predicted_class, confidence, "Success"

def predict_batch(img_paths, model, class_names, confidence_threshold=0.5):
    """
    Prediksi batch images
    """
    results = []
    
    for img_path in img_paths:
        predicted_class, confidence, status = predict_single_image(
            img_path, model, class_names, confidence_threshold
        )
        results.append({
            'image': img_path,
            'predicted_class': predicted_class,
            'confidence': confidence,
            'status': status
        })
    
    return pd.DataFrame(results)

## 9. Test Prediction dengan Sample Images

In [ ]:
def visualize_predictions(img_paths, model, class_names, num_samples=6):
    """
    Visualisasi prediksi untuk beberapa sample images
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for idx, img_path in enumerate(img_paths[:num_samples]):
        img_original = cv2.imread(img_path)
        img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)
        
        predicted_class, confidence, status = predict_single_image(
            img_path, model, class_names, confidence_threshold=0.3
        )
        
        axes[idx].imshow(img_original)
        axes[idx].axis('off')
        title = f"Pred: {predicted_class}\nConf: {confidence:.3f}"
        axes[idx].set_title(title, fontsize=10)
    
    plt.tight_layout()
    plt.show()

import glob

sample_jawa_paths = glob.glob(f"{JAWA_VAL_DIR}/ba/*.png")[:6]
if sample_jawa_paths:
    print("Sample predictions for Aksara Jawa:")
    visualize_predictions(sample_jawa_paths, model_jawa, class_names_jawa)

sample_sunda_paths = glob.glob(f"{SUNDA_DIR}/a/*.jpg")[:6]
if sample_sunda_paths:
    print("\nSample predictions for Aksara Sunda:")
    visualize_predictions(sample_sunda_paths, model_sunda, class_names_sunda)

## 10. Save & Load Model Functions

In [ ]:
def save_class_names(class_names, filename):
    """
    Save class names ke file
    """
    import json
    with open(filename, 'w') as f:
        json.dump(class_names, f)
    print(f"Class names saved to {filename}")

def load_class_names(filename):
    """
    Load class names dari file
    """
    import json
    with open(filename, 'r') as f:
        class_names = json.load(f)
    return class_names

def load_trained_model(model_path, class_names_path):
    """
    Load trained model dan class names
    """
    model = keras.models.load_model(model_path)
    class_names = load_class_names(class_names_path)
    print(f"Model loaded from {model_path}")
    print(f"Classes: {class_names}")
    return model, class_names

save_class_names(class_names_jawa, 'class_names_jawa.json')
save_class_names(class_names_sunda, 'class_names_sunda.json')

## 11. Example: Load Model dan Predict

Contoh cara menggunakan model yang sudah di-save untuk prediksi:

In [ ]:
loaded_model_jawa, loaded_class_names_jawa = load_trained_model(
    MODEL_JAWA_PATH, 
    'class_names_jawa.json'
)

sample_img_path = glob.glob(f"{JAWA_VAL_DIR}/ba/*.png")[0]
predicted_class, confidence, status = predict_single_image(
    sample_img_path, 
    loaded_model_jawa, 
    loaded_class_names_jawa, 
    confidence_threshold=0.5
)

print(f"\nTest prediction:")
print(f"Image: {sample_img_path}")
print(f"Predicted class: {predicted_class}")
print(f"Confidence: {confidence:.4f}")
print(f"Status: {status}")

---

## Pipeline Summary

### Pipeline Terbaik untuk OCR Aksara Jawa & Sunda:

**1. Preprocessing:**
- Konversi ke grayscale
- Gaussian blur untuk noise reduction
- Otsu thresholding untuk binarisasi
- Resize ke 224x224 pixels
- Normalisasi piksel [0, 1]

**2. Model Architecture:**
- Base: EfficientNetB0 (pre-trained ImageNet)
- Transfer learning dengan 2 fase:
  - Fase 1: Frozen base model, train classification head
  - Fase 2: Fine-tuning dengan unfrozen top layers
- Custom head: GlobalAvgPool + Dense(256) + Dropout + BatchNorm
- Optimizer: Adam dengan learning rate scheduling

**3. Training Strategy:**
- Dataset Jawa: Menggunakan struktur train/val yang sudah ada
- Dataset Sunda: Split 80:20 untuk train/val
- Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
- Batch size: 32, Initial LR: 0.001

**4. Post-processing:**
- Confidence thresholding (default: 0.5)
- Reject predictions dengan confidence rendah
- Classification report dan confusion matrix untuk evaluasi

**5. Model Deployment:**
- Save model dalam format .h5
- Save class names dalam JSON
- Load & predict function untuk inference

**Keunggulan Pipeline:**
- Transfer learning memanfaatkan pre-trained weights
- Two-phase training untuk konvergensi optimal
- Preprocessing yang robust untuk variasi dataset
- Confidence-based filtering untuk prediksi yang reliable
- Model terpisah untuk masing-masing aksara (specialized)